# DDL y DML

Este notebook se encarga de definir y poblar la base de datos.

In [ ]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv("/home/jovyan/work/.env", override=True)

print(f"Inicializando base de datos [{os.getenv('DB_NAME')}]...")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)
cursor = conn.cursor()

# Orden lógico de ejecución
archivos_sql_normales = [
    "01_ddl_schema.sql",
    "02_dml_datos.sql",
    "05_vistas.sql",
]

for archivo in archivos_sql_normales:
    ruta = f"/home/jovyan/work/sql/{archivo}"
    print(f"Ejecutando {archivo}...")
    
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = f.read()
        
    # Separar por punto y coma
    statements = [s.strip() for s in contenido.split(";") if s.strip()]
        
    for i, stmt in enumerate(statements):
        try:
            cursor.execute(stmt)
        except Exception as e:
            print("="*50)
            print(f"Error en {archivo} (Sentencia {i}):\n{e}")
            print("-"*50)
            print(stmt)
            print("="*50)
            break
    conn.commit()


archivos_sql_delim = [
    "03_procedures.sql", 
    "04_triggers.sql"
]

for archivo in archivos_sql_delim:
    ruta = f"/home/jovyan/work/sql/{archivo}"
    print(f"Ejecutando {archivo}...")
    
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = f.read()
    
    # Borrar delimiter
    contenido = contenido.replace("DELIMITER $$", "").replace("DELIMITER ;", "")
        
    # Separar por punto y coma
    statements = [s.strip() for s in contenido.split("$$") if s.strip()]
        
    for i, stmt in enumerate(statements):
        try:
            cursor.execute(stmt)
        except Exception as e:
            print("="*50)
            print(f"Error en {archivo} (Sentencia {i}):\n{e}")
            print("-"*50)
            print(stmt)
            print("="*50)
            break
    conn.commit()


cursor.close()
conn.close()
print("Base de datos inicializada exitosamente.")